<a href="https://colab.research.google.com/github/najahsadmim/llm-diagnostic-syncophancy/blob/main/LLM_Syncophancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Evidence-Grounded Mitigation of Diagnostic Sycophancy in Large Language Models through Confidence Calibration and Threshold-Based Decision Making**

## *Independent Research by NAJAH SADMIM*


##**Import Libraries**

In [26]:
#install libraries
%pip -q install -U openai anthropic pydantic

In [27]:
import openai
import anthropic
import pydantic

print("OpenAI:", openai.__version__)
print("Anthropic:", anthropic.__version__)
print("Pydantic:", pydantic.__version__)
print("Core imports successful.")

OpenAI: 3.27.0
Anthropic: 1.12.1
Pydantic: 2.14.0
Core imports successful.


In [28]:
#import libraries
import os
import re
import json
import random
import zipfile
import warnings

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
    )

# from sklearn.calibration import (
#     CalibratedClassiferCV,
#     calibration_curve
# )

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from pydantic import BaseModel, Field, ValidationError

from openai import OpenAI
from anthropic import Anthropic
from google import genai

warnings.filterwarnings("default")

SEED=42

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##**Load Dataset**

###**Import Data**

In [29]:
#authorize google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
#define paths

DATA_DIR= Path("/content/drive/MyDrive/DDXData")
DATASET_PATH= DATA_DIR / "DDXPlus_30K_Stratified_Research_Dataset.xlsx"
CONDITION_PATH= DATA_DIR / "release_conditions.json"
EVIDENCE_PATH= DATA_DIR / "release_evidences.json"

required_files={
    "30K Dataset": DATASET_PATH,
    "Conditions Metadata": CONDITION_PATH,
    "Evidence Metadata": EVIDENCE_PATH
}

for name, path in required_files.items():
  print(f"{name}: {'Found' if path.exists() else 'NOT FOUND'}")
  print(f"Path: {path}")
  print("-" * 60)

30K Dataset: Found
Path: /content/drive/MyDrive/DDXData/DDXPlus_30K_Stratified_Research_Dataset.xlsx
------------------------------------------------------------
Conditions Metadata: Found
Path: /content/drive/MyDrive/DDXData/release_conditions.json
------------------------------------------------------------
Evidence Metadata: Found
Path: /content/drive/MyDrive/DDXData/release_evidences.json
------------------------------------------------------------


In [31]:
#load 30k dataset

df=pd.read_excel(DATASET_PATH)

print("Dataset loaded successfully)")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully)
Rows: 30,000
Columns: 20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
#load conditions metadata

with open(CONDITION_PATH, "r", encoding="utf-8") as file:
  conditions_data=json.load(file)

print("Condition metadata loaded successfully.")
print("Metadata type:", type(conditions_data).__name__)
print("Number of top-level entries:", len(conditions_data))

Condition metadata loaded successfully.
Metadata type: dict
Number of top-level entries: 49


In [33]:
#load evidence metadata

with open(EVIDENCE_PATH, "r", encoding="utf-8") as file:
  evidences_data=json.load(file)

print("Evidence metadata loaded successfully.")
print("Metadata type:", type(evidences_data).__name__)
print("Number of evidence entries:", len(evidences_data))

Evidence metadata loaded successfully.
Metadata type: dict
Number of evidence entries: 223


##**Data Preprocessing**

###**Inspect Dataset**

In [34]:
#dataset inspection

print("Dataset dimensions:", df.shape)

Dataset dimensions: (30000, 20)


In [35]:
print("Column names:")
display(df.columns.tolist())

Column names:


['Case_ID',
 'Original_split',
 'Source_row_index',
 'PATHOLOGY',
 'AGE',
 'SEX',
 'EVIDENCES',
 'INITIAL_EVIDENCE',
 'DIFFERENTIAL_DIAGNOSIS',
 'Evidence_count',
 'Num_alternative_diagnoses',
 'Has_plausible_differential',
 'Differential_alternatives',
 'Condition_category',
 'DDXPlus_severity_level',
 'Urgency_flag',
 'ICD10_ID',
 'Age_band',
 'Evidence_count_band',
 'Experimental_subset']

In [36]:
print("First five rows:")
display(df.head())

First five rows:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Case_ID,Original_split,Source_row_index,PATHOLOGY,AGE,SEX,EVIDENCES,INITIAL_EVIDENCE,DIFFERENTIAL_DIAGNOSIS,Evidence_count,Num_alternative_diagnoses,Has_plausible_differential,Differential_alternatives,Condition_category,DDXPlus_severity_level,Urgency_flag,ICD10_ID,Age_band,Evidence_count_band,Experimental_subset
0,DDX_TRN_0005368,train,5368,Acute COPD exacerbation / infection,74,M,"['E_31', 'E_66', 'E_72', 'E_77', 'E_79', 'E_123', 'E_125', 'E_198', 'E_201', 'E_204_@_V_10', 'E_214']",E_201,"[['Acute COPD exacerbation / infection', 0.1244213435480098], ['Bronchitis', 0.11632363163469966], ['Bronchiectasis'...",11,15,Yes,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...,Lower respiratory,3,Unclear,j44.1,65+,Low (<=15),Development (train)
1,DDX_TRN_0005487,train,5487,Acute COPD exacerbation / infection,43,M,"['E_31', 'E_72', 'E_79', 'E_123', 'E_198', 'E_200', 'E_201', 'E_204_@_V_10', 'E_214']",E_201,"[['Bronchitis', 0.4055977749341997], ['Acute COPD exacerbation / infection', 0.3372112440871281], ['Anaphylaxis', 0....",9,3,Yes,Bronchitis; Anaphylaxis; Scombroid food poisoning,Lower respiratory,3,Unclear,j44.1,35-49,Low (<=15),Development (train)
2,DDX_TRN_0005825,train,5825,Acute COPD exacerbation / infection,54,M,"['E_31', 'E_66', 'E_72', 'E_77', 'E_79', 'E_123', 'E_125', 'E_199', 'E_200', 'E_201', 'E_204_@_V_10', 'E_214']",E_201,"[['Acute COPD exacerbation / infection', 0.1244213435480098], ['Bronchitis', 0.11632363163469966], ['Bronchiectasis'...",12,15,Yes,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...,Lower respiratory,3,Unclear,j44.1,50-64,Low (<=15),Development (train)
3,DDX_TRN_0007398,train,7398,Acute COPD exacerbation / infection,54,F,"['E_66', 'E_77', 'E_79', 'E_125', 'E_198', 'E_200', 'E_201', 'E_204_@_V_10']",E_201,"[['Bronchitis', 0.11184370918186151], ['Bronchiectasis', 0.10442882982979017], ['Pneumonia', 0.10167771803224472], [...",8,15,Yes,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Pulmonary embolism; Guillain-Barré syndrome; Bronchospasm...,Lower respiratory,3,Unclear,j44.1,50-64,Low (<=15),Development (train)
4,DDX_TRN_0013995,train,13995,Acute COPD exacerbation / infection,70,M,"['E_31', 'E_66', 'E_77', 'E_79', 'E_123', 'E_125', 'E_198', 'E_199', 'E_200', 'E_204_@_V_10', 'E_214']",E_66,"[['Acute COPD exacerbation / infection', 0.1244213435480098], ['Bronchitis', 0.11632363163469966], ['Bronchiectasis'...",11,15,Yes,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...,Lower respiratory,3,Unclear,j44.1,65+,Low (<=15),Development (train)


In [37]:
print("Data types:")
display(df.dtypes.to_frame(name="Data Type"))

Data types:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Data Type
Case_ID,object
Original_split,object
Source_row_index,int64
PATHOLOGY,object
AGE,int64
SEX,object
EVIDENCES,object
INITIAL_EVIDENCE,object
DIFFERENTIAL_DIAGNOSIS,object
Evidence_count,int64


In [38]:
print("Random sample of five cases:")
display(df.sample(5, random_state=42))

Random sample of five cases:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Case_ID,Original_split,Source_row_index,PATHOLOGY,AGE,SEX,EVIDENCES,INITIAL_EVIDENCE,DIFFERENTIAL_DIAGNOSIS,Evidence_count,Num_alternative_diagnoses,Has_plausible_differential,Differential_alternatives,Condition_category,DDXPlus_severity_level,Urgency_flag,ICD10_ID,Age_band,Evidence_count_band,Experimental_subset
2308,DDX_TRN_0623567,train,623567,Acute otitis media,62,F,"['E_53', 'E_54_@_V_161', 'E_54_@_V_192', 'E_55_@_V_130', 'E_56_@_3', 'E_57_@_V_123', 'E_58_@_6', 'E_59_@_6', 'E_91',...",E_201,"[['Bronchitis', 0.4245338451073684], ['Acute otitis media', 0.3666106410892656], ['Tuberculosis', 0.1843531138186945...",13,3,Yes,Bronchitis; Tuberculosis; Pneumonia,Upper respiratory / ENT,4,Non-urgent / less acute,H66.90,50-64,Low (<=15),Development (train)
22404,DDX_TRN_0569032,train,569032,Pulmonary embolism,106,F,"['E_34', 'E_45', 'E_53', 'E_54_@_V_192', 'E_55_@_V_55', 'E_55_@_V_127', 'E_55_@_V_128', 'E_55_@_V_159', 'E_55_@_V_17...",E_45,"[['Pulmonary embolism', 0.09460998666102163], ['Pulmonary neoplasm', 0.08421236894021357], ['Pericarditis', 0.076185...",25,17,Yes,Pulmonary neoplasm; Pericarditis; Acute pulmonary edema; Spontaneous pneumothorax; Panic attack; Possible NSTEMI / S...,Cardiovascular,2,Potentially urgent,i26,65+,High (>=25),Development (train)
23397,DDX_TRN_0086993,train,86993,SLE,24,M,"['E_3', 'E_53', 'E_54_@_V_161', 'E_54_@_V_192', 'E_55_@_V_26', 'E_55_@_V_34', 'E_55_@_V_35', 'E_55_@_V_74', 'E_55_@_...",E_129,"[['Possible NSTEMI / STEMI', 0.10045704382588468], ['SLE', 0.09941133832078523], ['Pulmonary embolism', 0.0895462763...",26,16,Yes,Possible NSTEMI / STEMI; Pulmonary embolism; Pancreatic neoplasm; Anemia; Scombroid food poisoning; Guillain-Barré s...,Autoimmune / systemic inflammatory,4,Non-urgent / less acute,M32,18-34,High (>=25),Development (train)
25058,DDX_TST_0044407,test,44407,Scombroid food poisoning,23,F,"['E_51', 'E_66', 'E_92', 'E_129', 'E_130_@_V_156', 'E_131_@_V_10', 'E_132_@_3', 'E_133_@_V_30', 'E_133_@_V_31', 'E_1...",E_66,"[['Scombroid food poisoning', 0.1556849245885445], ['Anaphylaxis', 0.1193902782826389], ['Atrial fibrillation', 0.11...",18,10,Yes,Anaphylaxis; Atrial fibrillation; PSVT; Acute pulmonary edema; Myocarditis; Guillain-Barré syndrome; Acute dystonic ...,Allergic / toxic reaction,2,Potentially urgent,T61.1,18-34,Mid (16-24),Final test (lock)
2664,DDX_TST_0079903,test,79903,Acute otitis media,9,F,"['E_1', 'E_53', 'E_54_@_V_161', 'E_55_@_V_129', 'E_56_@_9', 'E_57_@_V_123', 'E_58_@_10', 'E_59_@_1', 'E_114', 'E_204...",E_114,"[['Acute otitis media', 1.0]]",11,0,No,NaN,Upper respiratory / ENT,4,Non-urgent / less acute,H66.90,0-17,Low (<=15),Final test (lock)


In [39]:
print("Missing Values:")
display(df.isnull().sum())

Missing Values:


,0
Case_ID,0
Original_split,0
Source_row_index,0
PATHOLOGY,0
AGE,0
SEX,0
EVIDENCES,0
INITIAL_EVIDENCE,0
DIFFERENTIAL_DIAGNOSIS,0
Evidence_count,0


In [40]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [41]:
print("Target Variable Distribution:")
display(df["PATHOLOGY"].value_counts())

Target Variable Distribution:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
PATHOLOGY,
Anemia,1000
URTI,1000
Viral pharyngitis,1000
HIV (initial infection),803
Anaphylaxis,780
Localized edema,780
Pulmonary embolism,774
Influenza,760
Bronchitis,754


In [42]:
print("Disease Category Distribution:")
display(df["Condition_category"].value_counts())

Disease Category Distribution:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
Condition_category,
Upper respiratory / ENT,6376
Cardiovascular,5647
Lower respiratory,3926
Infectious (named pathogen / systemic),3113
Neurological,2508
Gastrointestinal,1919
Allergic / toxic reaction,1380
Autoimmune / systemic inflammatory,1134
Oncologic,1133


In [43]:
print("Age Statistics:")
display(df["AGE"].describe())

Age Statistics:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,AGE
count,30000.000000
mean,39.416333
std,23.089495
min,0.000000
25%,22.000000
50%,38.000000
75%,56.000000
max,109.000000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [44]:
print("Sex Distribution:")
display(df["SEX"].value_counts())

Sex Distribution:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
SEX,
F,15423
M,14577


In [45]:
df["Evidence_Count"] = df["EVIDENCES"].apply(len)

print("Evidence Count Statistics:")
display(df["Evidence_Count"].describe())

Evidence Count Statistics:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Evidence_Count
count,30000.000000
mean,238.044667
std,113.544709
min,24.000000
25%,157.000000
50%,243.000000
75%,320.000000
max,546.000000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [46]:
print("Evidence Metadata Sample:")
display(pd.DataFrame(evidences_data.items()).head())

Evidence Metadata Sample:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,0,1
0,E_91,"{'name': 'E_91', 'code_question': 'E_91', 'question_fr': 'Avez-vous objectivé ou ressenti de la fièvre?', 'question_..."
1,E_55,"{'name': 'E_55', 'code_question': 'E_53', 'question_fr': 'Avez-vous de la douleur quelque part?', 'question_en': 'Do..."
2,E_53,"{'name': 'E_53', 'code_question': 'E_53', 'question_fr': 'Avez-vous de la douleur à quelque part en lien avec votre ..."
3,E_57,"{'name': 'E_57', 'code_question': 'E_53', 'question_fr': 'Est-ce que la douleur se propage vers un autre endroit?', ..."
4,E_54,"{'name': 'E_54', 'code_question': 'E_53', 'question_fr': 'Caractérisez votre douleur:', 'question_en': 'Characterize..."


In [47]:
print("Initial Evidence Distribution:")
display(df["INITIAL_EVIDENCE"].value_counts())

Initial Evidence Distribution:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
INITIAL_EVIDENCE,
E_53,5358
E_66,2801
E_201,2101
E_181,1168
E_91,1146
...,...
E_178,39
E_145,37
E_23,35


In [48]:
print("Missing Differential Alternatives:")
print(df["Differential_alternatives"].isnull().sum())

print("\nSample Differential Alternatives:")
display(df["Differential_alternatives"].head())

Missing Differential Alternatives:
2120

Sample Differential Alternatives:


,Differential_alternatives
0,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...
1,Bronchitis; Anaphylaxis; Scombroid food poisoning
2,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...
3,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Pulmonary embolism; Guillain-Barré syndrome; Bronchospasm...
4,Bronchitis; Bronchiectasis; Pneumonia; Pulmonary neoplasm; Guillain-Barré syndrome; Atrial fibrillation; Myocarditis...


In [49]:
print("Evidence Count Distribution:")
display(df["Evidence_Count"].value_counts().sort_index())

Evidence Count Distribution:


,count
Evidence_Count,
24,4
25,16
31,1
32,7
33,20
...,...
524,1
530,1
531,2


In [50]:
print("Dataset Columns:")
display(df.dtypes.to_frame(name="Data Type"))

print("\nUnique Diagnoses:", df["PATHOLOGY"].nunique())
print("Unique Sex Values:", df["SEX"].nunique())

Dataset Columns:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Data Type
Case_ID,object
Original_split,object
Source_row_index,int64
PATHOLOGY,object
AGE,int64
SEX,object
EVIDENCES,object
INITIAL_EVIDENCE,object
DIFFERENTIAL_DIAGNOSIS,object
Evidence_count,int64



Unique Diagnoses: 49
Unique Sex Values: 2


##**Feature Selection**

##**Feature Engineering**

##**Train, Test and Split**

##**Self-Diagnosis Experiment Construction**